In [ ]:
import os
import csv
import sys
from datetime import datetime

csv_folder_path = "../arquivos/"
sql_folder_path = "../q2/"
sql_file_name = "schema.sql"
tables = [] 
table_with_foreign_keys = {}

def get_target_table(col_name):
    # Remove '_id' e retorna o nome da tabela associado.
    # Example: 'customer_id' -> 'customers'
    prefix = col_name.replace('_id', '')
    
    if prefix.endswith('y'):
        return prefix[:-1] + 'ies'  # company_id -> companies
    elif prefix.endswith(('s', 'ch', 'sh', 'x', 'z')):
        return prefix + 'es'        # bus_id -> buses
    else:
        return prefix + 's'         # customer_id -> customers

def infer_data_type(value):
    value = value.strip().lower()
    if not value or value.lower() in ("null", "none", "na", "n/a"):
        return None 

    timestamp_formats = [
        "%Y-%m-%d %H:%M:%S",        
        "%Y-%m-%dT%H:%M:%S",        
        "%Y-%m-%d %H:%M:%S.%f",     
        "%d/%m/%Y %H:%M:%S",        
        "%d/%m/%Y %H:%M",           
    ]
    for fmt in timestamp_formats:
        try:
            datetime.strptime(value, fmt)
            return "TIMESTAMP" 
        except ValueError:
            pass
    
    date_formats = ["%Y-%m-%d", "%d/%m/%Y", "%m/%d/%Y"]
    for fmt in date_formats:
        try:
            datetime.strptime(value, fmt)
            return "DATE" 
        except ValueError:
            pass
    try:
        int_value = int(value)
        return "INTEGER" 
    except ValueError:
        pass

    if value.lower() in ("true", "false"):
        return "BOOLEAN" 

    try:
        float(value)
        return "FLOAT" 
    except ValueError:
        pass

    return "TEXT" 

def generate_sql_from_csvs(csv_folder_path, sample_rows=1000):
    sql_schema = ""
    if not os.path.isdir(csv_folder_path):
        print(f"Error: A pasta '{csv_folder_path}' não existe. Corrija o caminho da pasta.")
        return

    type_priority = {
        "BOOLEAN": 1, 
        "INTEGER": 2, 
        "FLOAT": 3, 
        "DATE": 4, 
        "TIMESTAMP": 5, 
        "TEXT": 6
    }
    
    for file_name in os.listdir(csv_folder_path):
        if file_name.endswith(".csv"):
            file_path = os.path.join(csv_folder_path, file_name)
            table_name = os.path.splitext(file_name)[0]
            tables.append(table_name)

            try:
                with open(file_path, mode="r", encoding="utf-8") as file:
                    reader = csv.reader(file)
                    if(os.path.getsize(file_path) == 0): 
                        raise Exception(f"Arquivo vazio.\n")
                    
                    headers = next(reader)  
                    col_types = {col_name: None for col_name in headers if col_name} 
                    
                    for row_idx, row in enumerate(reader):
                        if row_idx >= sample_rows:
                            break
                        for index, value in enumerate(row):
                            if index >= len(headers):
                                break
                            
                            col_name = headers[index]
                            if not col_name:
                                continue

                            current_type = infer_data_type(value)
                            
                            if current_type is not None:
                                existing_type = col_types[col_name]
                                if not existing_type:
                                    col_types[col_name] = current_type
                                else:
                                    if type_priority[current_type] > type_priority[existing_type]:
                                        col_types[col_name] = current_type

                for col_name in col_types:
                        if col_types[col_name] is None:
                            col_types[col_name] = "TEXT"

                columns = []
                foreing_keys = []
                foreing_keys_target_tables = []
                for col_name in headers:
                    if col_name:
                        if col_name == "id":
                            columns.append(f"    {col_name} {col_types[col_name]} PRIMARY KEY")
                        else:
                            if col_name.endswith("_id"):
                                foreing_keys.append(col_name)
                                foreing_keys_target_tables.append(get_target_table(col_name))
                            columns.append(f"    {col_name} {col_types[col_name]}")
                
                if foreing_keys:
                    table_with_foreign_keys[table_name] = {"foreing_keys": foreing_keys, "target_tables": foreing_keys_target_tables}
                
                if columns:
                    sql_schema += f"CREATE TABLE IF NOT EXISTS {table_name} (\n"
                    sql_schema += ",\n".join(columns)
                    sql_schema += "\n);\n\n"

            except Exception as e:
                print(f"Erro ao ler o arquivo '{file_name}': {e}\n")
                sys.exit(1)
                
    # --- BLOCO CORRIGIDO DE GERALÇÃO DE FOREIGN KEYS ---
    if table_with_foreign_keys:
        for table_name, fkeys_dict in table_with_foreign_keys.items():
            col_names = fkeys_dict["foreing_keys"]
            target_tables = fkeys_dict["target_tables"]
            
            # Filtra apenas as chaves cujas tabelas destino realmente existem no mapeamento de CSVs
            valid_constraints = []
            for i in range(len(col_names)):
                if target_tables[i] in tables:
                    constraint_str = (
                        f"    ADD CONSTRAINT fk_{table_name}_{target_tables[i]}\n"
                        f"        FOREIGN KEY ({col_names[i]}) REFERENCES {target_tables[i]}(id)"
                    )
                    valid_constraints.append(constraint_str)
            
            # Se houver chaves válidas para essa tabela, monta o comando agrupado corretamente
            if valid_constraints:
                sql_schema += f"ALTER TABLE {table_name}\n"
                sql_schema += ",\n".join(valid_constraints)  # Separa multiplas FKs por VÍRGULA
                sql_schema += ";\n\n"                      # Finaliza a tabela com PONTO E VÍRGULA

    os.makedirs(sql_folder_path, exist_ok=True)
    
    with open(os.path.join(sql_folder_path, sql_file_name), "w", encoding="utf-8") as file:
        file.write(sql_schema)
    print(f"Arquivo '{sql_file_name}' gerado com sucesso\nno caminho '{os.path.join(sql_folder_path, sql_file_name)}'\n")

# Executa a função
generate_sql_from_csvs(csv_folder_path)


#fix employees_id type to be text